# exp_7 · 00 protocol — freeze, tests, rechecks (IDSSE only)
Freezes `PREREG.md` (SHA-256 + code-derived constants + SHA-256 of every reused file and of exp_7's token caches)
before any exp_7 fit or scoring, records the write guard, runs the synthetic tests, then (after freezing) scores
`exp3_ref_ens3` from saved weights on exp_7's views through exp_6's path: the `target_like`, measured, stress and
clean 25 / 50 % rows must equal exp_6's stored rows (binding); the AR(1) rows are report only.

In [1]:
from pathlib import Path
import os, sys, json, time, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp7_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_7')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp7_common as C
import exp1_common as C1
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')
def make_log(name):
    path = C.LOG_DIR / f'{name}_progress.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    def log(message, **_):
        print(message, flush=True)
        with path.open('a') as handle:
            handle.write(f'{now()} {message}\n')
    return log

if not C.PROTOCOL.exists():
    C.freeze_prereg()
    print('PREREG frozen now (before any exp_7 fit or scoring)')
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])

before = C.guard_record(C.RESULTS / 'write_guard_before.json')
print('write guard recorded:', {k: v['files'] for k, v in before['folders'].items()})
print('public check spent by exp_1 (never repeated):', C.public_check_spent())
pre = C1.read_json(C.RESULTS / 'prefreeze_checks.json')
print('pre-freeze:', {k: v for k, v in pre.items() if k != 'unsimulated_target_aug_equals_exp2'})

PREREG frozen now (before any exp_7 fit or scoring)


PREREG verified 189874eb7d34ab4c | frozen 2026-11-11T01:29:38+00:00


write guard recorded: {'exp_0': 42630, 'exp_1': 215, 'exp_2': 312, 'exp_3': 407, 'exp_4': 638, 'exp_5': 247, 'exp_6': 81, 'data': 12, 'data_metric_repo': 1}
public check spent by exp_1 (never repeated): True
pre-freeze: {'checked_at_utc': '2026-11-11T01:26:46+00:00', 'token_cache_seconds': 86.6, 'token_cache_files': 49, 'token_cache_mb': 834.8, 'all_unsimulated_equal': True, 'fingerprints': {'exp7_robust': '6fbb3c58f313', 'exp7_robust_s1': '101817ff9e86', 'exp7_robust_s2': 'e5cee02bdc44'}, 'tiny_synthetic_fit_seconds': 0.89, 'write_guard_unchanged': True}


In [2]:
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests'],
                        cwd=ROOT, capture_output=True, text=True, env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'})
summary = [line for line in result.stdout.strip().splitlines() if 'passed' in line or 'failed' in line][-1]
C1.write_json(C.RESULTS / 'tests.json', {'run_at_utc': now(), 'returncode': result.returncode, 'summary': summary})
print(summary)
if result.returncode:
    raise RuntimeError('exp_7 synthetic tests failed')

15 passed, 1 warning in 14.91s


In [3]:
import exp7_eval as E
log = make_log('00')
rows = E.reference_rows(('target_like', 'measured', 'stress', 'clean25', 'clean50', 'ar1'), log)
pd.DataFrame(rows).to_csv(C.RESULTS / 'ref_rows.csv', index=False)
recheck = E.recheck_against_exp6(rows)
C1.write_json(C.RESULTS / 'rechecks.json', {'checked_at_utc': now(), 'reference_vs_exp6_rows': recheck})
print(recheck)
if not recheck['all_equal']:
    raise RuntimeError('Binding recheck failed: exp_7 views do not reproduce exp_6 reference rows')
table = pd.DataFrame(rows).groupby(['view', 'label_set'])[['spell_f1', 'count_error_ratio']].mean().round(4)
print(table)

REF target_like J03WMX


REF measured J03WMX


REF stress J03WMX


REF clean25 J03WMX


REF clean50 J03WMX


REF ar1 J03WMX


... (33 outputs left out)

REF clean25 J03WR9


REF clean50 J03WR9


REF ar1 J03WR9


{'target_like': {'rows': 14, 'max_abs_diff': 1.1102230246251565e-16, 'equal': True}, 'measured': {'rows': 14, 'max_abs_diff': 6.938893903907228e-17, 'equal': True}, 'stress': {'rows': 14, 'max_abs_diff': 9.020562075079397e-17, 'equal': True}, 'clean25': {'rows': 14, 'max_abs_diff': 1.1102230246251565e-16, 'equal': True}, 'clean50': {'rows': 14, 'max_abs_diff': 1.1102230246251565e-16, 'equal': True}, 'all_equal': True}
                       spell_f1  count_error_ratio
view        label_set                             
ar1         ctrl         0.6400             0.1436
            provider     0.5692             0.0790
clean25     ctrl         0.7669            -0.0982
            provider     0.6824            -0.1859
clean50     ctrl         0.7817            -0.0821
            provider     0.6991            -0.1696
measured    ctrl         0.6032             0.2353
            provider     0.5396             0.1657
stress      ctrl         0.6391             0.0960
            provi